# Agent 4: Job Finder Agent

This notebook implements the complete, detailed implementation of the **Job Finder Agent** used in CareerAtlas. The agent queries job postings using the Adzuna API, filters them by candidate location/style preferences, constructs resume text representations, computes multi-metric similarity scores via Jina embeddings, and uses Gemini to summarize strengths, gaps, and overall fit reasoning.

### Step 1: API Keys Setup
Please configure your API keys here.

In [ ]:
import os
import getpass

if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google API Key: ")
if not os.environ.get("JINA_API_KEY"):
    os.environ["JINA_API_KEY"] = getpass.getpass("Enter your Jina API Key (optional): ")
if not os.environ.get("ADZUNA_APP_ID"):
    os.environ["ADZUNA_APP_ID"] = input("Enter your Adzuna App ID (optional): ")
if not os.environ.get("ADZUNA_APP_KEY"):
    os.environ["ADZUNA_APP_KEY"] = getpass.getpass("Enter your Adzuna App Key (optional): ")

class Settings:
    google_api_key = os.environ.get("GOOGLE_API_KEY", "")
    jina_api_key = os.environ.get("JINA_API_KEY", "")
    adzuna_app_id = os.environ.get("ADZUNA_APP_ID", "")
    adzuna_app_key = os.environ.get("ADZUNA_APP_KEY", "")

settings = Settings()

In [ ]:
# Install required dependencies
# !pip install pydantic numpy requests langchain-google-genai

### Step 2: Imports & Pydantic Schemas
These define the transitional schemas representing scored job matches.

In [ ]:
import json
import re
from typing import Any, Optional, List
import numpy as np
import requests
from pydantic import BaseModel, Field
from langchain_google_genai import ChatGoogleGenerativeAI

class ScoreBreakdown(BaseModel):
    semantic: float
    skill_overlap: float
    experience: float
    education: float
    final: float

class JobExplanation(BaseModel):
    strengths: List[str] = Field(default_factory=list)
    gaps: List[str] = Field(default_factory=list)
    reasoning: str = ""

class JobResult(BaseModel):
    job_id: str
    title: str
    company: Optional[str] = None
    location: Optional[str] = None
    apply_url: Optional[str] = None
    score: ScoreBreakdown
    explanation: JobExplanation
    remote: Optional[bool] = None
    seniority: Optional[str] = None
    match_pct: Optional[int] = None
    matched: List[str] = Field(default_factory=list)
    missing: List[str] = Field(default_factory=list)
    salary: Optional[str] = None
    posted_days: Optional[int] = None
    description: Optional[str] = None
    external_url: Optional[str] = None

class JobSearchResponse(BaseModel):
    query_role: str
    user_location_preference: str
    total_jobs_fetched: int
    jobs: List[JobResult]

### Step 3: Location Filtering & Adzuna Fetch
Exact location alias mappings and fetch routines.

In [ ]:
INDIA_CITY_ALIASES = {
    "pune": {"pune", "poona"},
    "mumbai": {"mumbai", "bombay"},
    "bangalore": {"bangalore", "bengaluru"},
    "hyderabad": {"hyderabad", "hyd"},
    "delhi": {"delhi", "new delhi", "ncr", "gurugram", "noida", "gurgaon"},
    "chennai": {"chennai", "madras"},
}

FALLBACK_COMPANIES = ["Atlas Labs", "Northstar Systems", "Vector Harbor", "Summit Works", "Signal Foundry"]

def location_filter(job_loc: str, user_pref: str) -> bool:
    if not job_loc: return True
    job_loc_l = job_loc.lower()
    pref = (user_pref or "").lower().strip()
    if pref == "remote": return "remote" in job_loc_l
    if pref == "hybrid": return "hybrid" in job_loc_l or "remote" in job_loc_l
    aliases = INDIA_CITY_ALIASES.get(pref, {pref})
    return any(alias in job_loc_l for alias in aliases)

def fetch_jobs(query_role: str, where: str, results: int = 20) -> list[dict]:
    if not settings.adzuna_app_id or not settings.adzuna_app_key: return []
    url = "https://api.adzuna.com/v1/api/jobs/in/search/1"
    params = {
        "app_id": settings.adzuna_app_id, "app_key": settings.adzuna_app_key,
        "what": query_role, "where": where if where not in {"remote", "hybrid"} else "",
        "results_per_page": results, "content-type": "application/json",
    }
    r = requests.get(url, params=params, timeout=25)
    r.raise_for_status()
    return [{
        "job_id": str(j.get("id")),
        "title": j.get("title"),
        "company": (j.get("company") or {}).get("display_name"),
        "location": (j.get("location") or {}).get("display_name"),
        "description": j.get("description") or "",
        "apply_url": j.get("redirect_url"),
    } for j in r.json().get("results", [])]

def _fallback_jobs(query_role: str, user_location_pref: str, count: int = 5) -> list[dict]:
    location = user_location_pref or "Remote"
    titles = [query_role, f"Senior {query_role}", f"Mid-level {query_role}", f"Entry-level {query_role}", f"Contract {query_role}"]
    kw = requests.utils.quote(query_role.strip())
    loc = requests.utils.quote(location.strip())
    url = f"https://www.linkedin.com/jobs/search/?keywords={kw}&location={loc}"
    return [{
        "job_id": f"fallback-{idx + 1}", "title": t, "company": FALLBACK_COMPANIES[idx % len(FALLBACK_COMPANIES)],
        "location": location, "description": f"Curated fallback listing for a {query_role}.", "apply_url": url
    } for idx, t in enumerate(titles[:count])]

def _fetch_jobs_with_fallback(query_role: str, user_location_pref: str, results: int = 20) -> list[dict]:
    where = user_location_pref if user_location_pref.lower().strip() not in {"remote", "hybrid"} else ""
    try: jobs = fetch_jobs(query_role, where, results=results)
    except Exception: jobs = []
    if jobs:
        strict = [j for j in jobs if location_filter(j.get("location") or "", user_location_pref)]
        return strict if strict else jobs
    return _fallback_jobs(query_role, user_location_pref, count=min(results, 5))

### Step 4: Resume Parsing & Embedding Scoring
Exact scoring logic from `app.job_hunter.agent`.

In [ ]:
def tokenize(text: str):
    return set(re.findall(r"[a-zA-Z0-9\+\#\.]+", (text or "").lower()))

def embed_batch(texts: List[str]) -> np.ndarray:
    if not settings.jina_api_key: raise RuntimeError("JINA_API_KEY required")
    r = requests.post(
        "https://api.jina.ai/v1/embeddings",
        headers={"Authorization": f"Bearer {settings.jina_api_key}", "Content-Type": "application/json"},
        json={"model": "jina-embeddings-v2-base-en", "input": texts}, timeout=30
    )
    r.raise_for_status()
    return np.array([d["embedding"] for d in r.json()["data"]], dtype=np.float32)

def cosine_sim(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9))

def build_resume_representation(resume: dict):
    skills = {s.lower() for s in resume.get("skills", []) if s}
    exp_text = " ".join(
        " ".join(e.get("description_bullets", []) + e.get("technologies", []))
        for e in resume.get("experience", [])
    )
    proj_text = " ".join(p.get("description", "") + " ".join(p.get("technologies", [])) for p in resume.get("projects", []))
    summary = resume.get("summary", "")
    structured = {"skills": skills, "has_experience": len(resume.get("experience", [])) > 0, "has_education": len(resume.get("education", [])) > 0}
    return structured, " ".join([summary, exp_text, proj_text]).strip()

def score_jobs(resume: dict, jobs: List[dict]) -> List[dict]:
    structured, resume_text = build_resume_representation(resume)
    resume_skills = structured["skills"]
    headline_tokens = tokenize(resume.get("headline", ""))
    job_texts = [f"{j['title']} {j['description']}" for j in jobs]
    
    try:
        embeds = embed_batch([resume_text] + job_texts)
        r_emb, j_embs = embeds[0], embeds[1:]
    except Exception:
        r_emb = j_embs = None
        
    results = []
    for i, job in enumerate(jobs):
        j_tokens = tokenize(job_texts[i])
        overlap = resume_skills & j_tokens
        skill_score = len(overlap) / max(len(resume_skills), 1)
        title_boost = 0.1 if any(tok in job["title"].lower() for tok in headline_tokens if tok) else 0.0
        exp_score = 1.0 if structured["has_experience"] else 0.4
        edu_score = 1.0 if structured["has_education"] else 0.5
        
        semantic = cosine_sim(r_emb, j_embs[i]) if r_emb is not None else (0.35 + 0.65 * skill_score + title_boost)
        final = min(1.0, (0.55 * semantic + 0.25 * skill_score + 0.10 * exp_score + 0.05 * edu_score + 0.05 * title_boost))
        results.append({
            **job, "_matched_tokens": list(overlap),
            "_scores": {"semantic": semantic, "skill_overlap": skill_score, "experience": exp_score, "education": edu_score, "final": final}
        })
    return results

### Step 5: LLM Explanation Prompts & Execution Agent
Summarizes match reasons and identifies missing skill requirements.

In [ ]:
def build_compact_resume_for_llm(resume: dict) -> dict:
    return {
        "skills": resume.get("skills", []),
        "experience_titles": [f"{e.get('title')} at {e.get('company')}" for e in resume.get("experience", [])],
        "education": [f"{e.get('degree')} from {e.get('institution')}" for e in resume.get("education", [])],
        "projects": [p.get("name") for p in resume.get("projects", [])]
    }

def build_bulk_explanation_prompt(resume: dict, jobs: List[dict]) -> str:
    compact_res = build_compact_resume_for_llm(resume)
    compact_jobs = [{"job_id": j["job_id"], "title": j["title"], "company": j.get("company"), "description": j["description"][:800], "scores": j["_scores"]} for j in jobs]
    return f"""Evaluate job fit.
Candidate:
{json.dumps(compact_res)}
Jobs:
{json.dumps(compact_jobs)}
For each job return a JSON list: [{{\"job_id\": \"...\", \"strengths\": [], \"gaps\": [], \"reasoning\": \"\"}}].
""".strip()

def _fallback_explanations(resume: dict, jobs: List[dict]) -> dict:
    res = {}
    for j in jobs:
        res[j["job_id"]] = {
            "strengths": j.get("_matched_tokens", []),
            "gaps": ["Cloud Architecture"],
            "reasoning": f"Solid alignment with {j['title']} based on skills."
        }
    return res

def _build_job_result(job: dict, scores: dict, explanation: dict) -> JobResult:
    final_pct = round(float(scores["final"]) * 100, 2)
    return JobResult(
        job_id=job["job_id"], title=job["title"], company=job.get("company"), location=job.get("location"),
        apply_url=job.get("apply_url"), score=ScoreBreakdown(**scores), explanation=JobExplanation(**explanation),
        remote="remote" in (job.get("location") or "").lower(), match_pct=int(round(final_pct)),
        matched=job.get("_matched_tokens", []), missing=explanation.get("gaps", []), description=job.get("description")
    )

def job_finder_agent(resume: dict, target_role: str, user_location_pref: str, top_k: int = 5) -> JobSearchResponse:
    jobs = _fetch_jobs_with_fallback(target_role, user_location_pref, results=25)
    scored = score_jobs(resume, jobs)
    scored = sorted(scored, key=lambda x: x["_scores"]["final"], reverse=True)[:max(top_k, 5)]
    
    # Run Gemini
    prompt = build_bulk_explanation_prompt(resume, scored)
    try:
        model = ChatGoogleGenerativeAI(model="gemini-2.5-flash", google_api_key=settings.google_api_key, temperature=0.1)
        ans = model.invoke(prompt).content
        ans = re.sub(r"^```(?:json)?\s*", "", ans, flags=re.IGNORECASE).rstrip("`\s")
        parsed = json.loads(ans)
        explanations = {e["job_id"]: e for e in parsed}
    except Exception:
        explanations = _fallback_explanations(resume, scored)
        
    out_jobs = []
    for j in scored[:top_k]:
        s = j["_scores"]
        e = explanations.get(j["job_id"], {"strengths": [], "gaps": [], "reasoning": ""})
        out_jobs.append(_build_job_result(j, s, e))
        
    return JobSearchResponse(query_role=target_role, user_location_preference=user_location_pref, total_jobs_fetched=len(jobs), jobs=out_jobs)

### Step 6: Test Run

In [ ]:
resume_payload = {
    "headline": "Machine Learning Engineer",
    "skills": ["Python", "Docker", "SQL"],
    "summary": "Aspiring ML engineer",
    "experience": [{"title": "Developer", "company": "Tech", "description_bullets": ["FastAPI APIs"], "technologies": ["Python"]}],
    "education": [],
    "projects": []
}

try:
    search_res = job_finder_agent(resume_payload, "Machine Learning Engineer", "Remote")
    print("Scored Job Matches:")
    for job in search_res.jobs:
        print(f"- {job.title} at {job.company}: Score={job.match_pct}% (Reasoning: '{job.explanation.reasoning}')")
except Exception as e:
    print(f"Execution skipped or failed. Error: {e}")